<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 55px">    


# Fundamentals of Chaining
###  Multiple Chains

---

### Learning objective:
By the end of this lesson, you will be able to practice chaining and multiple chaining.


### About:  
In this lesson students will supercharge their workflows by processing documents in a model and creating multiple chains. 


### Prerequisites:
- Python (required) 
- Intro to LangChain and prior prompt  eng. lessons (required) 
- Visual Studio Code (recommended)
- GitHub Copilot lessons (recommended) 

### Contents
1. [Imports](#imports)
1. [LCEL chains](#lcel)
1. [Multiple chains](#chains)



<a id='imports'></a>
## Imports

In [2]:
#basic chains
from langchain_openai import ChatOpenAI #openai chatbot
from langchain_core.prompts import ChatPromptTemplate #template for chat prompts
from langchain_core.output_parsers import StrOutputParser #output parser for string output 

#documents 
from langchain_core.documents import Document
from langchain.chains.combine_documents import create_stuff_documents_chain

<a id='lcel'></a>
## Chain Review 
**LCEL** allows you build complex chains with runnable components, "runnables" greatly reducing the amount of code you need to write for common model tasks. They come with common methods or interfaces that you can use in many situations. You quickly switch out components and create chains of chains (multiple chains). 

#### Basic Chain
- Language or chat model 
- Prompt Template
- Output parser


#### Key Interfaces
- **Invoke (invoke):** Pass in a string and get back a string
- **Stream (stream):** Streams the response 
- **Batch (batch):** Pass in multiple prompts and get back the response 
- **Async (ainvoke):** Run asynchronously 

#### Docs
1. [LangChain LCEL](https://python.langchain.com/docs/expression_language/)
1. [LangChain Interface](https://python.langchain.com/docs/expression_language/interface)

<a id='chains'></a>
## Multiple chains
Automate your work flows and build more complex systems by creating chains of chains. 



In [ ]:
# Example 
model = ChatOpenAI(model_name="gpt-4o-mini")
# model = ChatOpenAI(openai_api_key="...", model_name="gpt-4")

#chain 1 prompt and chain to output a city
prompt1 = ChatPromptTemplate.from_template("Name a popular city for tourists to visit in {month}.")
chain1 = prompt1 | model | StrOutputParser()

#chain 2 prompt and chain- uses city from prompt 1 to generate destinations 
prompt2 = ChatPromptTemplate.from_template(
    "What country is the {city} in? Name 3 popular tourist destinations in that country."
)
chain2 = (
    {"city": chain1} #include chain 1! 
    | prompt2
    | model
    | StrOutputParser()
)

chain2.invoke({"month": "August"})

<a id='trouble'></a>
## Troubleshooting
As we add more complexity to our chains, it will be important to see the mechanics behind LangChain to understand the results we are getting, to improve our prompts, and debug. 

- Review tools in LangChain (e.g., debug and verbose) to help track logs 
Demo each of debug and verbose 

#### Debug import and code: 
```python
from langchain.globals import set_debug

set_debug(True)
```

### Let's revist our prior example 

In [4]:
# Example chain setup 

model = ChatOpenAI( model_name="gpt-4o-mini")
# model = ChatOpenAI(openai_api_key="...", model_name="gpt-4o-mini")

#chain 1 prompt and chain to output a city
prompt1 = ChatPromptTemplate.from_template("Name a popular city for tourists to visit in {month}.")
chain1 = prompt1 | model | StrOutputParser()

#chain 2 prompt and chain- uses city from prompt 1 to generate destinations 
prompt2 = ChatPromptTemplate.from_template(
    "What country is the {city} in? Name 3 popular tourist destinations in that country."
)
chain2 = (
    {"city": chain1} #include chain 1! 
    | prompt2
    | model
    | StrOutputParser()
)

### Let's invoke the chain with debug turned on

Read through each line. 
#### Take note of: 
- What city did the model generate in chain 1? 
- What was passed to chain 2? 


In [ ]:
from langchain.globals import set_debug #import 
set_debug(True)
chain2.invoke({"month": "August"})

In [6]:
# Great! Now let's turn off debug for now...
set_debug(False)


<a id='lab'></a>
## TRY It!

Build a series of 3 chains that 1) loads family preferences recommends a vacation destination, 2) it passes that vacation spot to a second model that recommends the best month to travel, 3) based on that month generates a 3-day travel itinerary. 

chain|chain type| input | output
--|--|--| --|
chain1 | create_stuff_documents_chain | family_docs | vacation destination 
chain2 | custom basic chain | vacation destination | month
chain3 | custom basic chain | month | 3-day itinerary

#### Family Travel Documents 

In [7]:
# Generated by GPT
family_docs = [
    Document(page_content="Sarah likes to travel to warm places but not cold ones"),
    Document(page_content="John prefers cities with a rich history but doesn't like crowded places"),
    Document(page_content="Emma loves beach vacations but is not a fan of hiking"),
    Document(page_content="Mike enjoys adventurous trips but doesn't like long flights from Wales")
]

#### Set Model

In [8]:
model = ChatOpenAI(openai_api_key="...", model_name="gpt-4o-mini")
# model = ChatOpenAI(openai_api_key="...", model_name="gpt-4")

### Chain 1
Use create_stuff_documents_chain to load family_docs and generate a vacation destination 

In [9]:
# add your code here!
prompt = ChatPromptTemplate.from_messages(
    [("system", "What city should this family travel:\n\n{context}.")]
)
llm = ChatOpenAI( model_name="gpt-4o-mini")
# llm = ChatOpenAI(openai_api_key="...", model_name="gpt-4")
chain1 = create_stuff_documents_chain(llm, prompt)

# build iteratively and test along the way 
# chain1.invoke({"context": family_docs})

### Chain 2
Build a new chain to pass that vacation destination to a model generate the best month to travel to this location

In [10]:
# add your code here!
#chain 1 prompt and chain to output a city
prompt2 = ChatPromptTemplate.from_template(
    "Recommend a month to travel to this {city}"
)
chain2 = (
    {"city": chain1} #include chain 1! 
    | prompt2
    | model
    | StrOutputParser()
)

# build iteratively and test along the way 
# chain2.invoke({"context": family_docs})

### Chain 3
Build a 3rd and final chain that generates a 3-day travel itinerary for the family 

In [ ]:
# add your code here!
prompt3 = ChatPromptTemplate.from_template(
    "Recommend a 3-day travel itinerary with one destination each day in {city} during {month}"
)
chain3 = (
    {"city": chain2, "month": chain2} #include chain 1! 
    | prompt3
    | model
    | StrOutputParser()
)

# build iteratively and test along the way 
chain3.invoke({"context": family_docs})

In [ ]:
# With debug
set_debug(True)
chain3.invoke({"context": family_docs})